***CASED MODEL***

In [ ]:
import json
import torch
import pickle
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    get_scheduler
)
from torch.optim import AdamW # Corrected import statement
from sklearn.model_selection import train_test_split
from pathlib import Path

# ================================================
# CONFIGURATION — only these lines change
# ================================================

DATASET_PATH = "/content/drive/MyDrive/DataSet/LatestProperDatasets/!LearningNER.json"
OUTPUT_DIR   = "/content/drive/MyDrive/Models/EVA/ner_bert_cased"
MODEL_NAME   = "bert-base-cased"   # ← cased — preserves capitalization
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 2e-5
VAL_SPLIT    = 0.2
SEED         = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# User-defined LABELS
LABELS = [
    "O",
    "B-date",         "I-date",
    "B-day",          "I-day",
    "B-duration",     "I-duration",
    "B-new_date",     "I-new_date",
    "B-new_duration", "I-new_duration",
    "B-new_priority", "I-new_priority",
    "B-new_status",   "I-new_status",
    "B-new_taskname", "I-new_taskname",
    "B-new_time",     "I-new_time",
    "B-person",       "I-person",
    "B-priority",     "I-priority",
    "B-status",       "I-status",
    "B-task_name",    "I-task_name",
    "B-time",         "I-time"
]
# LABELS = [
#     "O",
#     "B-topic",      "I-topic",
#     "B-subtopic",   "I-subtopic",
#     "B-start_date", "I-start_date",
#     "B-duration",   "I-duration",
#     "B-person",     "I-person",
#     "B-date",       "I-date",
# ]
# ================================================
# LOAD DATASET
# ================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total examples: {len(data)}")

# ================================================
# PREPROCESSING: Convert raw text and entities to tokens and BIO labels
# ================================================

def create_tokens_and_labels(text, entities):
    words = text.split() # Simple whitespace tokenizer for word-level tokens
    labels = ["O"] * len(words)

    # Sort entities by start position to handle overlaps correctly
    # Assuming 'value' is the text span of the entity
    # This part needs to be robust to handle cases where entity value is not exact match

    for entity in entities:
        entity_value = entity["value"]
        entity_type  = entity["entity"]

        # Find the starting word index of the entity_value in words
        # This is a simplified approach, a more robust solution might use token offsets
        start_idx = -1
        for i in range(len(words)):
            current_span = []
            for j in range(i, len(words)):
                current_span.append(words[j])
                if " ".join(current_span) == entity_value:
                    start_idx = i
                    end_idx = j
                    break
            if start_idx != -1:
                break

        if start_idx != -1:
            labels[start_idx] = f"B-{entity_type}"
            for i in range(start_idx + 1, end_idx + 1):
                labels[i] = f"I-{entity_type}"

    return words, labels

processed_data = []
for item in data:
    tokens, labels = create_tokens_and_labels(item["text"], item["entities"])
    processed_data.append({
        "text": item["text"], # Keep original text for reference
        "tokens": tokens,
        "labels": labels
    })

data = processed_data

# ================================================
# BUILD LABEL MAP
# ================================================

# Use the user-defined LABELS list
label_list = LABELS
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

print(f"Labels ({num_labels}): {label_list}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT_DIR}/label_map.pkl", "wb") as f:
    pickle.dump({"label2id": label2id, "id2label": id2label}, f)

# ================================================
# TOKENIZER — BertTokenizerFast instead of DistilBert
# ================================================

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# ================================================
# DATASET CLASS — identical to Model 1
# ================================================

class NERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len):
        self.data      = data
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item   = self.data[idx]
        tokens = item["tokens"]
        labels = item["labels"]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        word_ids          = encoding.word_ids()
        label_ids         = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(self.label2id[labels[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "token_type_ids" : encoding["token_type_ids"].squeeze(),  # BERT needs this
            "labels"         : torch.tensor(label_ids, dtype=torch.long)
        }

# ================================================
# SPLIT
# ================================================

train_data, val_data = train_test_split(
    data, test_size=VAL_SPLIT,
    random_state=SEED
)

train_dataset = NERDataset(train_data, tokenizer, label2id, MAX_LEN)
val_dataset   = NERDataset(val_data,   tokenizer, label2id, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# ================================================
# MODEL — BertForTokenClassification instead of DistilBert
# ================================================

model = BertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(DEVICE)

# ================================================
# OPTIMIZER AND SCHEDULER — identical
# ================================================

optimizer   = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# ================================================
# TRAIN AND EVAL — token_type_ids added for BERT
# ================================================

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids        = batch["input_ids"].to(DEVICE)
        attention_mask   = batch["attention_mask"].to(DEVICE)
        token_type_ids   = batch["token_type_ids"].to(DEVICE)
        labels           = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids        = batch["input_ids"].to(DEVICE)
            attention_mask   = batch["attention_mask"].to(DEVICE)
            token_type_ids   = batch["token_type_ids"].to(DEVICE)
            labels           = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=2)

            for pred_seq, label_seq in zip(preds, labels):
                for p, l in zip(pred_seq, label_seq):
                    if l.item() != -100:
                        all_preds.append(id2label[p.item()])
                        all_labels.append(id2label[l.item()])

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    acc     = correct / len(all_labels) if all_labels else 0

    return total_loss / len(loader), acc, all_preds, all_labels


# ================================================
# TRAINING LOOP — identical
# ================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
best_val_acc = 0

print("\nStarting NER training (bert-base-cased)...")
print("=" * 60)

for epoch in range(EPOCHS):
    train_loss              = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f} | Val Acc : {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

    print("-" * 60)

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Device: cuda
Total examples: 12292
Labels (29): ['O', 'B-date', 'I-date', 'B-day', 'I-day', 'B-duration', 'I-duration', 'B-new_date', 'I-new_date', 'B-new_duration', 'I-new_duration', 'B-new_priority', 'I-new_priority', 'B-new_status', 'I-new_status', 'B-new_taskname', 'I-new_taskname', 'B-new_time', 'I-new_time', 'B-person', 'I-person', 'B-priority', 'I-priority', 'B-status', 'I-status', 'B-task_name', 'I-task_name', 'B-time', 'I-time']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 0.9997)
------------------------------------------------------------
Epoch 4/10
  Train Loss : 0.0023
  Val Loss   : 0.0016 | Val Acc : 0.9997


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 0.9997)
------------------------------------------------------------


In [3]:
import torch
import pickle
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification
)

# ================================================
# CONFIGURATION
# ================================================

MODEL_DIR = "/content/drive/MyDrive/Models/EVA/ner_bert_cased"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ================================================
# LOAD MODEL
# ================================================

tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)
model     = BertForTokenClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

with open(f"/content/drive/MyDrive/Models/EVA/ner_bert_cased/label_map.pkl", "rb") as f:
    label_map = pickle.load(f)

id2label = label_map["id2label"]
print(f"Labels: {list(label_map['label2id'].keys())}")

# ================================================
# INFERENCE FUNCTION
# ================================================

def predict_entities(text: str) -> dict:
    # NO lowercase — cased model preserves capitalization
    tokens = text.split()

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids        = encoding["input_ids"].to(DEVICE)
    attention_mask   = encoding["attention_mask"].to(DEVICE)
    token_type_ids   = encoding["token_type_ids"].to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

    preds    = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
    word_ids = encoding.word_ids()

    # align predictions back to original tokens
    token_preds = {}
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx not in token_preds:
            token_preds[word_idx] = id2label[preds[idx]]

    # build entities from BIO tags
    entities = []
    current  = None

    for i, token in enumerate(tokens):
        label = token_preds.get(i, "O")

        if label.startswith("B-"):
            if current:
                entities.append(current)
            current = {
                "value" : token,
                "entity": label[2:]
            }
        elif label.startswith("I-") and current:
            current["value"] += " " + token
        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    return {
        "text"    : text,
        "tokens"  : tokens,
        "entities": entities
    }

# ================================================
# TEST
# ================================================

test_inputs = [
    # learning with go through pattern
    "remind me for data science quiz tomorrow 5pm",
]

print("\nNER Results (bert-base-cased):")
print("=" * 60)
for text in test_inputs:
    result = predict_entities(text)
    print(f"Input    : {result['text']}")
    if result["entities"]:
        for ent in result["entities"]:
            print(f"  [{ent['entity']}] → {ent['value']}")
    else:
        print("  No entities found")
    print("-" * 40)

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Labels: ['O', 'B-date', 'I-date', 'B-day', 'I-day', 'B-duration', 'I-duration', 'B-new_date', 'I-new_date', 'B-new_duration', 'I-new_duration', 'B-new_priority', 'I-new_priority', 'B-new_status', 'I-new_status', 'B-new_taskname', 'I-new_taskname', 'B-new_time', 'I-new_time', 'B-person', 'I-person', 'B-priority', 'I-priority', 'B-status', 'I-status', 'B-task_name', 'I-task_name', 'B-time', 'I-time']

NER Results (bert-base-cased):
Input    : remind me for data science quiz tomorrow 5pm
  [task_name] → data science quiz
  [date] → tomorrow
  [time] → 5pm
----------------------------------------


***distilbert***

In [ ]:
import json
import torch
import pickle
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForTokenClassification,
    get_scheduler
)
from torch.optim import AdamW # Corrected import statement
from sklearn.model_selection import train_test_split
from pathlib import Path

# ================================================
# CONFIGURATION
# ================================================

DATASET_PATH = "/content/drive/MyDrive/DataSet/LatestProperDatasets/!LearningNER.json"
OUTPUT_DIR   = "/content/drive/MyDrive/Models/EVA/distilbert_ner"
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 2e-5
VAL_SPLIT    = 0.2
SEED         = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ================================================
# LOAD DATASET
# ================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total examples: {len(data)}")

# ================================================
# PREPROCESSING: Convert raw text and entities to tokens and BIO labels
# ================================================

def create_tokens_and_labels(text, entities):
    words = text.split() # Simple whitespace tokenizer for word-level tokens
    labels = ["O"] * len(words)

    # Sort entities by start position to handle overlaps correctly
    # Assuming 'value' is the text span of the entity
    # This part needs to be robust to handle cases where entity value is not exact match

    for entity in entities:
        entity_value = entity["value"]
        entity_type  = entity["entity"]

        # Find the starting word index of the entity_value in words
        # This is a simplified approach, a more robust solution might use token offsets
        start_idx = -1
        for i in range(len(words)):
            current_span = []
            for j in range(i, len(words)):
                current_span.append(words[j])
                if " ".join(current_span) == entity_value:
                    start_idx = i
                    end_idx = j
                    break
            if start_idx != -1:
                break

        if start_idx != -1:
            labels[start_idx] = f"B-{entity_type}"
            for i in range(start_idx + 1, end_idx + 1):
                labels[i] = f"I-{entity_type}"

    return words, labels

processed_data = []
for item in data:
    tokens, labels = create_tokens_and_labels(item["text"], item["entities"])
    processed_data.append({
        "text": item["text"], # Keep original text for reference
        "tokens": tokens,
        "labels": labels
    })

data = processed_data

# ================================================
# BUILD LABEL MAP
# ================================================

all_labels = set()
for item in data:
    for label in item["labels"]:
        all_labels.add(label)

label_list = sorted(all_labels)
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

print(f"Labels ({num_labels}): {label_list}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT_DIR}/label_map.pkl", "wb") as f:
    pickle.dump({"label2id": label2id, "id2label": id2label}, f)

# ================================================
# TOKENIZER
# ================================================

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# ================================================
# DATASET CLASS
# ================================================

class NERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len):
        self.data      = data
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item   = self.data[idx]
        tokens = item["tokens"]
        labels = item["labels"]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        word_ids    = encoding.word_ids()
        label_ids   = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(self.label2id[labels[word_idx]])
            else:
                # sub-tokens of same word get -100
                label_ids.append(-100)
            previous_word_idx = word_idx

        return {
            "input_ids"      : encoding["input_ids"].squeeze(),
            "attention_mask" : encoding["attention_mask"].squeeze(),
            "labels"         : torch.tensor(label_ids, dtype=torch.long)
        }

# ================================================
# SPLIT
# ================================================

train_data, val_data = train_test_split(
    data, test_size=VAL_SPLIT,
    random_state=SEED
)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

train_dataset = NERDataset(train_data, tokenizer, label2id, MAX_LEN)
val_dataset   = NERDataset(val_data,   tokenizer, label2id, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ================================================
# MODEL
# ================================================

model = DistilBertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(DEVICE)

# ================================================
# OPTIMIZER AND SCHEDULER
# ================================================

optimizer   = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# ================================================
# TRAIN AND EVAL
# ================================================

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=2)

            for pred_seq, label_seq in zip(preds, labels):
                for p, l in zip(pred_seq, label_seq):
                    if l.item() != -100:
                        all_preds.append(id2label[p.item()])
                        all_labels.append(id2label[l.item()])

    # token level accuracy
    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    acc     = correct / len(all_labels) if all_labels else 0

    return total_loss / len(loader), acc, all_preds, all_labels


# ================================================
# TRAINING LOOP
# ================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
best_val_acc = 0

print("\nStarting NER training...")
print("=" * 60)

for epoch in range(EPOCHS):
    train_loss              = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f} | Val Acc : {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

    print("-" * 60)

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Device: cuda
Total examples: 12292
Labels (26): ['B-date', 'B-day', 'B-duration', 'B-new_date', 'B-new_duration', 'B-new_priority', 'B-new_status', 'B-new_taskname', 'B-new_time', 'B-person', 'B-priority', 'B-status', 'B-task_name', 'B-time', 'I-date', 'I-duration', 'I-new_date', 'I-new_duration', 'I-new_status', 'I-new_taskname', 'I-new_time', 'I-person', 'I-status', 'I-task_name', 'I-time', 'O']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 9833 | Val: 2459


... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 0.9992)
------------------------------------------------------------
Epoch 4/10
  Train Loss : 0.0053
  Val Loss   : 0.0022 | Val Acc : 0.9998


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model saved (val_acc: 0.9998)
------------------------------------------------------------
Epoch 5/10
  Train Loss : 0.0033
  Val Loss   : 0.0017 | Val Acc : 0.9997
------------------------------------------------------------
Epoch 6/10
  Train Loss : 0.0028
  Val Loss   : 0.0016 | Val Acc : 0.9998
------------------------------------------------------------
Epoch 7/10
  Train Loss : 0.0027
  Val Loss   : 0.0017 | Val Acc : 0.9997
------------------------------------------------------------
Epoch 8/10
  Train Loss : 0.0020
  Val Loss   : 0.0014 | Val Acc : 0.9997
------------------------------------------------------------
Epoch 9/10
  Train Loss : 0.0018
  Val Loss   : 0.0017 | Val Acc : 0.9997
------------------------------------------------------------
Epoch 10/10
  Train Loss : 0.0016
  Val Loss   : 0.0014 | Val Acc : 0.9998
------------------------------------------------------------

Best Val Accuracy: 0.9998


In [ ]:
import torch
import pickle
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForTokenClassification
)

# ================================================
# CONFIGURATION
# ================================================

MODEL_DIR = "/content/drive/MyDrive/Models/EVA/distilbert_ner/"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ================================================
# LOAD MODEL
# ================================================

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
model     = DistilBertForTokenClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

with open(f"{MODEL_DIR}label_map.pkl", "rb") as f:
    label_map = pickle.load(f)

id2label = label_map["id2label"]
print(f"Labels: {list(label_map['label2id'].keys())}")

# ================================================
# INFERENCE FUNCTION
# ================================================

def predict_entities(text: str) -> dict:
    # lowercase for distilbert uncased
    tokens = text.lower().split()

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    preds    = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
    word_ids = encoding.word_ids()

    # align predictions back to original tokens
    token_preds = {}
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx not in token_preds:
            token_preds[word_idx] = id2label[preds[idx]]

    # build entities from BIO tags
    entities  = []
    current   = None

    for i, token in enumerate(tokens):
        label = token_preds.get(i, "O")

        if label.startswith("B-"):
            if current:
                entities.append(current)
            current = {
                "value" : token,
                "entity": label[2:]
            }
        elif label.startswith("I-") and current:
            current["value"] += " " + token
        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    return {
        "text"    : text,
        "tokens"  : tokens,
        "entities": entities
    }

# ================================================
# TEST
# ================================================

test_inputs = ["need to prepare physics exam on monday 5pm for 1h with hamid ali"]

print("\nNER Results (DistilBERT):")
print("=" * 60)
for text in test_inputs:
    result = predict_entities(text)
    print(f"Input    : {result['text']}")
    if result["entities"]:
        for ent in result["entities"]:
            print(f"  [{ent['entity']}] → {ent['value']}")
    else:
        print("  No entities found")
    print("-" * 40)

Device: cuda


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Labels: ['B-date', 'B-day', 'B-duration', 'B-new_date', 'B-new_duration', 'B-new_priority', 'B-new_status', 'B-new_taskname', 'B-new_time', 'B-person', 'B-priority', 'B-status', 'B-task_name', 'B-time', 'I-date', 'I-duration', 'I-new_date', 'I-new_duration', 'I-new_status', 'I-new_taskname', 'I-new_time', 'I-person', 'I-status', 'I-task_name', 'I-time', 'O']

NER Results (DistilBERT):
Input    : need to prepare physics exam on monday 5pm for 1h with hamid ali
  [task_name] → prepare
  [task_name] → physics exam
  [date] → monday
  [time] → 5pm
  [duration] → 1h
  [person] → hamid
  [person] → ali
----------------------------------------
